# Tokenize tool-use SFT traces with loss mask

Reads `tool_traces_grounded.jsonl`, builds the canonical text format, tokenizes with offset tracking, and emits per-token loss masks where:

- `0` for the prompt portion (`Question:`) and tool outputs (`Result:`)
- `1` for everything the model should learn to generate (`Thought:`, `Action:`, `Input:`, `Final:`, EOT)

Masking the `Result:` lines is critical — without it the model would learn to hallucinate tool outputs at inference time.

## 1. Setup

In [ ]:
!pip install -q tokenizers

In [ ]:
import os, json
from google.colab import drive
from tokenizers import Tokenizer

drive.mount('/content/drive')

SPARKYLLM = '/content/drive/MyDrive/sparkyllm'
TOKENIZER_PATH = os.path.join(SPARKYLLM, 'datasets_pretrain', 'tokenizer_out', 'tokenizer.json')
DATA_DIR  = os.path.join(SPARKYLLM, 'datasets_sft', 'tool_traces')
GROUNDED_FILE  = os.path.join(DATA_DIR, 'tool_traces_grounded.jsonl')
TOKENIZED_FILE = os.path.join(DATA_DIR, 'tool_traces_tokenized.jsonl')
META_FILE      = os.path.join(DATA_DIR, 'tokenize_meta.json')

BLOCK_SIZE = 768  # must match training

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
vocab_size = tokenizer.get_vocab_size()
if vocab_size % 64 != 0:
    vocab_size_padded = ((vocab_size + 63) // 64) * 64
else:
    vocab_size_padded = vocab_size
eot_id = tokenizer.token_to_id('<|endoftext|>')
print(f'Tokenizer vocab: {vocab_size} (padded: {vocab_size_padded}) | EOT: {eot_id}')

## 2. Tokenize-with-mask helper

Builds the full trace text by section, tokenizes the full text once with offsets, then walks the offsets to assign each token to its source section. This is robust to BPE quirks at section boundaries.

In [ ]:
def build_trace_pieces(trace):
    """Return list of (text, mask_value) sections in order."""
    pieces = []
    pieces.append((f"Question: {trace['question']}\n", 0))
    for step in trace['steps']:
        pieces.append((f"Thought: {step['thought']}\n", 1))
        pieces.append((f"Action: {step['action']}\n", 1))
        pieces.append((f"Input: {step['input']}\n", 1))
        pieces.append((f"Result: {step['result']}\n", 0))
    pieces.append((f"Final: {trace['final']}", 1))
    return pieces

def tokenize_with_mask(trace):
    """Returns (input_ids, loss_mask) or None if too long."""
    pieces = build_trace_pieces(trace)
    full_text = ''.join(p[0] for p in pieces)
    
    # Build (start, end, mask_val) sections in character offsets
    sections = []
    pos = 0
    for text, mv in pieces:
        sections.append((pos, pos + len(text), mv))
        pos += len(text)
    
    enc = tokenizer.encode(full_text)
    input_ids = list(enc.ids)
    loss_mask = []
    for tok_start, tok_end in enc.offsets:
        mv = 0
        for sec_start, sec_end, m in sections:
            if sec_start <= tok_start < sec_end:
                mv = m
                break
        loss_mask.append(mv)
    
    # Append EOT — model should learn to STOP at the end
    input_ids.append(eot_id)
    loss_mask.append(1)
    
    assert len(input_ids) == len(loss_mask)
    if len(input_ids) > BLOCK_SIZE + 1:
        return None
    return input_ids, loss_mask

## 3. Process all grounded traces

In [ ]:
from collections import Counter

if not os.path.exists(GROUNDED_FILE):
    raise FileNotFoundError(f'{GROUNDED_FILE} not found — run generate_tool_traces.ipynb first.')

written = 0
skipped_long = 0
skipped_bad = 0
lengths = []
mask_coverage = []
by_cat = Counter()

with open(TOKENIZED_FILE, 'w') as f_out, open(GROUNDED_FILE) as f_in:
    for line in f_in:
        try:
            trace = json.loads(line)
        except Exception:
            skipped_bad += 1
            continue
        if 'final' not in trace or not trace.get('steps'):
            skipped_bad += 1
            continue
        try:
            r = tokenize_with_mask(trace)
        except Exception as e:
            skipped_bad += 1
            continue
        if r is None:
            skipped_long += 1
            continue
        ids, mask = r
        obj = {
            'input_ids': ids,
            'loss_mask': mask,
            'category': trace.get('category', 'unknown'),
            'n_tokens': len(ids),
        }
        f_out.write(json.dumps(obj) + '\n')
        written += 1
        lengths.append(len(ids))
        mask_coverage.append(sum(mask) / len(mask))
        by_cat[trace.get('category', 'unknown')] += 1

print(f'Written:        {written}')
print(f'Skipped (long): {skipped_long}')
print(f'Skipped (bad):  {skipped_bad}')
if lengths:
    print(f'Length: min={min(lengths)}  median={sorted(lengths)[len(lengths)//2]}  max={max(lengths)}')
    print(f'Avg mask coverage: {sum(mask_coverage)/len(mask_coverage):.1%}')
print()
print('By category:')
for cat, n in sorted(by_cat.items()):
    print(f'  {cat:25s} {n}')

## 4. Save metadata

In [ ]:
meta = {
    'tokenizer_path': TOKENIZER_PATH,
    'vocab_size': vocab_size,
    'vocab_size_padded': vocab_size_padded,
    'eot_id': eot_id,
    'block_size': BLOCK_SIZE,
    'num_examples': written,
    'skipped_long': skipped_long,
    'skipped_bad': skipped_bad,
    'by_category': dict(by_cat),
    'avg_length': sum(lengths) / max(1, len(lengths)),
    'max_length': max(lengths) if lengths else 0,
    'avg_mask_coverage': sum(mask_coverage) / max(1, len(mask_coverage)),
}
with open(META_FILE, 'w') as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))
print()
print('Next: run sft_tool_use.ipynb')